# Day 2 Session 4 -- Exercises: Multi-Agent Workflows & Agentic Systems

Work through the exercises below to practice building multi-agent systems. You have the demos notebook as reference.

**Engineering context:** You are an engineer building multi-agent orchestration systems. Your users (consultants) work in engagement teams where a Partner supervises, Associates specialize by domain, and work flows across functional boundaries. Multi-agent architectures mirror these team dynamics -- supervisor routing, agent handoffs, parallel workstreams, and collaborative deliverable creation.

**Structure:**
- Exercise 1 (Required): Two-Agent Supervisor-Worker System
- Exercise 2 (Required): Three-Agent Handoff Pipeline
- Exercise 3 (Required): Collaborative Report Builder
- Optional Exercise 1 (Intermediate): Intent-Based Router with 4 Specialists
- Optional Exercise 2 (Intermediate): Parallel Analysis with Synthesis
- Optional Exercise 3 (Intermediate): Full Engagement Delivery System

## Setup

In [ ]:
!pip install -q langchain langchain-openai langchain-core langgraph python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Literal
import json
import os

print("All imports successful!")

## Recap

In the demos you saw five multi-agent patterns:
1. **Supervisor-Worker** -- A central supervisor routes tasks to specialized worker agents (Demo 1)
2. **Agent Handoff** -- Agents pass structured context to each other in sequence (Demo 2)
3. **Parallel Execution** -- Independent agents run concurrently, results merged by synthesis agent (Demo 3)
4. **Collaborative Deliverable** -- Multiple agents build on a shared output sequentially (Demo 4)
5. **End-to-End System** -- Intent classification + specialist routing + quality review (Demo 5)

The exercises below build these patterns progressively.

---
## Exercise 1 (Easy): Two-Agent Supervisor-Worker System

**Reference:** Demo 1 in the demos notebook

> **Hint:** Conditional routing uses `add_conditional_edges` -- the routing function returns a string that maps to the next node name.

**Scenario:** An Engagement Manager receives diverse client requests during a transformation engagement. Some requests require **quantitative analysis** (financial modeling, data analysis, benchmarking) while others require **strategic advisory** (market entry, competitive positioning, growth strategy). The EM must route each request to the right Associate and review the output before delivering to the client.

**Your task:** Build a system with:
- A **supervisor** node (Engagement Manager) that classifies requests as `"quantitative"` or `"strategic"` using the LLM
- A **quantitative** worker node that handles financial/data analysis with structured, data-driven output
- A **strategic** worker node that handles market/competitive analysis with hypothesis-driven recommendations
- A **review** node (Partner) that ensures the deliverable is client-ready

**Architecture:**
```
[supervisor] --conditional--> [quantitative] --> [review] --> END
             \--conditional--> [strategic]   --> [review] --> END
```

### Step 1: Define the State

Define a `Task1State` TypedDict with four fields:
- `task` (str): The incoming client request
- `assigned_to` (str): Will hold `"quantitative"` or `"strategic"` after supervisor classifies
- `worker_output` (str): The specialist's analysis
- `final_response` (str): The Partner-reviewed final deliverable

In [ ]:
# Step 1: Define state and initialize LLM (provided)

class Task1State(TypedDict):
    task: str
    assigned_to: str
    worker_output: str
    final_response: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

print("State and LLM ready.")

### Step 2: Define the Supervisor Node

The supervisor should:
- Use a SystemMessage instructing the LLM to classify as `"quantitative"` or `"strategic"`
- Pass the client request as a HumanMessage
- Extract the LLM's response, strip whitespace, lowercase it
- Return `{"assigned_to": <classification>}`

**Hint:** Look at `supervisor_node` in Demo 1 -- yours is simpler (only 2 categories instead of 3).

In [ ]:
# Step 2: Define supervisor node (provided)

def supervisor_node(state: Task1State) -> dict:
    """Engagement Manager classifies the request type."""
    response = llm.invoke([
        SystemMessage(content="You are an Engagement Manager. Classify the client request as "
                      "'quantitative' (financial modeling, data analysis, benchmarking) or "
                      "'strategic' (market entry, competitive positioning, growth strategy). "
                      "Respond with one word only."),
        HumanMessage(content=state["task"])
    ])
    assigned = response.content.strip().lower()
    print(f"  EM assigned to: {assigned}")
    return {"assigned_to": assigned}

### Step 3: Define the Worker Nodes

Define two worker functions:

**`quantitative_worker`** -- handles financial/data requests:
- SystemMessage: specialized in quantitative analysis, financial modeling, data-driven insights
- Returns: `{"worker_output": f"[Quantitative Analysis] {response.content}"}`

**`strategic_worker`** -- handles market/competitive requests:
- SystemMessage: specialized in strategy, market assessment, hypothesis-driven recommendations
- Returns: `{"worker_output": f"[Strategic Advisory] {response.content}"}`

In [ ]:
# Step 3: Define worker nodes

# TODO: Define quantitative_worker(state: Task1State) -> dict
# Hint: Use SystemMessage describing a quantitative analyst. Input = state["task"]
# Return {"worker_output": f"[Quantitative Analysis] {response.content}"}

def quantitative_worker(state: Task1State) -> dict:
    # YOUR CODE HERE
    pass

# TODO: Define strategic_worker(state: Task1State) -> dict
# Hint: Same pattern but with strategic advisory SystemMessage

def strategic_worker(state: Task1State) -> dict:
    # YOUR CODE HERE
    pass

### Step 4: Define the Review Node

The Partner reviews and finalizes:
- Takes both `state["task"]` and `state["worker_output"]` as input
- Adds executive perspective, ensures client-readiness
- Returns: `{"final_response": response.content}`

In [ ]:
# Step 4: Define review node (provided)

def review_node(state: Task1State) -> dict:
    """Partner reviews and finalizes the deliverable."""
    response = llm.invoke([
        SystemMessage(content="You are a Partner reviewing an Associate's deliverable. "
                      "Assess quality, add executive perspective, finalize for client delivery."),
        HumanMessage(content=f"Client Request: {state['task']}\n\nAssociate Deliverable:\n{state['worker_output']}")
    ])
    return {"final_response": response.content}

### Step 5: Define Routing + Build Graph

The routing function checks `state["assigned_to"]`:
- If it contains `"quantitative"` -> return `"quantitative"`
- Otherwise -> return `"strategic"`

Then build the graph:
1. Create `StateGraph(Task1State)`
2. Add all 4 nodes: `supervisor`, `quantitative`, `strategic`, `review`
3. Set entry point to `"supervisor"`
4. Add conditional edges from `"supervisor"` using your routing function
5. Add regular edges: both workers -> `"review"`, `"review"` -> `END`
6. Compile

In [ ]:
# Step 5: Define routing function + build graph

# Routing function (provided)
def route_task(state: Task1State) -> str:
    return "quantitative" if "quantitative" in state["assigned_to"] else "strategic"

# Build graph (provided -- uncomment after writing worker nodes)
# graph = StateGraph(Task1State)
# graph.add_node("supervisor", supervisor_node)
# graph.add_node("quantitative", quantitative_worker)
# graph.add_node("strategic", strategic_worker)
# graph.add_node("review", review_node)
# graph.set_entry_point("supervisor")
# graph.add_conditional_edges("supervisor", route_task, {
#     "quantitative": "quantitative",
#     "strategic": "strategic"
# })
# graph.add_edge("quantitative", "review")
# graph.add_edge("strategic", "review")
# graph.add_edge("review", END)
# app = graph.compile()

### Step 6: Test

Run with both request types to verify correct routing.

In [ ]:
# Step 6: Test with both request types

# TODO: Uncomment and run after completing the above steps
# for task in [
#     "Build a financial model to assess the ROI of consolidating three regional warehouses into one automated distribution center",
#     "Develop a market entry strategy for our client expanding into Southeast Asian consumer electronics"
# ]:
#     r = app.invoke({"task": task, "assigned_to": "", "worker_output": "", "final_response": ""})
#     print(f"\nClient Request: {task}")
#     print(f"Routed to: {r['assigned_to']}")
#     print(f"Partner-Reviewed Deliverable: {r['final_response'][:250]}...\n")


### Expected Output

When you run the test cell, you should see output similar to:

```
  EM assigned to: quantitative

Client Request: Build a financial model to assess the ROI of consolidating three regional
warehouses into one automated distribution center
Routed to: quantitative
Partner-Reviewed Deliverable: The financial analysis provides a solid framework for evaluating the
warehouse consolidation. The Associate has correctly identified the key cost drivers...

  EM assigned to: strategic

Client Request: Develop a market entry strategy for our client expanding into Southeast Asian
consumer electronics
Routed to: strategic
Partner-Reviewed Deliverable: The strategy work is well-structured with clear hypotheses around
market sizing, competitive dynamics, and channel strategy...
```

The exact wording will vary (LLM output is non-deterministic), but you should see:
- The supervisor correctly routing the financial request to `quantitative` and the market request to `strategic`
- Each worker producing domain-appropriate analysis
- The Partner review adding executive perspective and finalizing the output

---
## Exercise 2 (Easy): Three-Agent Handoff Pipeline

**Reference:** Demo 2 in the demos notebook

> **Hint:** Each chain step feeds its output as input to the next. Build and test each step independently first.

**Scenario:** A consulting engagement flows through three phases: **Research** (gather data and context), **Analysis** (interpret findings and develop insights), and **Recommendation** (formulate actionable advice). Each phase builds on the previous one's output, and structured handoff context ensures continuity.

**Your task:** Build a handoff pipeline:
- **Research Agent** -- gathers relevant data, market context, and background information
- **Analysis Agent** -- interprets the research findings, identifies patterns and insights
- **Recommendation Agent** -- formulates actionable recommendations based on the analysis

**Architecture:**
```
[research] --> [analysis] --> [recommendation] --> END
     |              |                |
     v              v                v
  handoff_1      handoff_2      final_output
```

**Key learning:** Each agent must pass a `handoff_context` field summarizing what was done and what the next agent should focus on.

In [ ]:
# Exercise 2 - Step 1: Define State

# TODO: Define Ex2State TypedDict with these fields:
#   - client_request (str): The original client question
#   - research_output (str): Research agent's findings
#   - analysis_output (str): Analysis agent's insights
#   - recommendation_output (str): Final recommendations
#   - handoff_context (str): Structured context passed between agents


# TODO: Initialize LLM
# llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.3")))


In [ ]:
# Exercise 2 - Step 2: Define the Three Agents

# TODO: Define research_agent(state: Ex2State) -> dict
#   - SystemMessage: "You are a McKinsey research analyst. Gather relevant data,
#     market context, industry trends, and background information for the client request.
#     Structure your output with: key data points, market context, and areas for deeper analysis."
#   - HumanMessage: state["client_request"]
#   - Create handoff_context: summarize what research was done and what analysis should focus on
#   - Print: f"  [Research] Findings delivered ({len(response.content)} chars)"
#   - Return: {"research_output": response.content, "handoff_context": <your context>}


# TODO: Define analysis_agent(state: Ex2State) -> dict
#   - SystemMessage: "You are a McKinsey analyst. Interpret research findings,
#     identify patterns, develop insights, and structure key conclusions.
#     Focus on the 'so what' -- what do these findings mean for the client?"
#   - HumanMessage: Include BOTH handoff_context AND research_output from state
#   - Update handoff_context with analysis summary
#   - Print: f"  [Analysis] Insights developed ({len(response.content)} chars)"
#   - Return: {"analysis_output": response.content, "handoff_context": <updated context>}


# TODO: Define recommendation_agent(state: Ex2State) -> dict
#   - SystemMessage: "You are a McKinsey senior consultant. Formulate clear, actionable
#     recommendations based on the analysis. Include: priority actions, timeline,
#     expected impact, and implementation considerations."
#   - HumanMessage: Include analysis_output and handoff_context
#   - Print: f"  [Recommendation] Advice formulated ({len(response.content)} chars)"
#   - Return: {"recommendation_output": response.content}


In [ ]:
# Exercise 2 - Step 3: Build Graph + Run

# TODO: Build StateGraph with 3 nodes in sequence:
#   research --> analysis --> recommendation --> END


# TODO: Test with this request:
# result = app.invoke({
#     "client_request": "Our client, a European mid-market bank, is losing deposits to digital-first neobanks. They need to understand the threat and respond.",
#     "research_output": "", "analysis_output": "", "recommendation_output": "", "handoff_context": ""
# })
# print(f"\nFinal Recommendations:\n{result['recommendation_output'][:400]}...")


### Expected Output

You should see three sequential agent executions, each building on the previous:

```
  [Research] Findings delivered (2800 chars)
  [Analysis] Insights developed (3100 chars)
  [Recommendation] Advice formulated (3500 chars)

Final Recommendations:
## Strategic Recommendations for European Mid-Market Bank

### Priority 1: Digital Experience Modernization (0-6 months)
- Launch mobile-first account opening with 5-minute onboarding
- Implement real-time notifications and spending insights...
```

Key verification:
- Each agent's output should reference findings from the previous stage
- The handoff_context should grow richer at each step
- The final recommendations should be specific and actionable, grounded in the research and analysis

---
## Exercise 3 (Easy): Collaborative Report Builder

**Reference:** Demo 4 in the demos notebook

> **Hint:** Each chain step feeds its output as input to the next. Build and test each step independently first.

**Scenario:** A client needs a board-ready report on entering a new market. Three roles collaborate:
- **Outline Agent** (Engagement Manager) -- creates the report structure and key messages
- **Content Agent** (Associate) -- develops detailed content for each section
- **Review Agent** (Partner) -- polishes for executive readiness

**Your task:** Build a collaborative pipeline where each agent builds upon the previous agent's work to produce a polished deliverable.

**Architecture:**
```
[outline_agent] --> [content_agent] --> [review_agent] --> END
       |                  |                   |
       v                  v                   v
    outline            draft              polished
```

**Key learning:** Unlike the handoff pattern (Exercise 2), here each agent is contributing to the SAME deliverable -- not passing independent findings forward.

In [ ]:
# Exercise 3 - Step 1: Define State

# TODO: Define Ex3State TypedDict with these fields:
#   - topic (str): The report topic
#   - outline (str): The EM's report structure
#   - draft (str): The Associate's full content
#   - polished (str): The Partner's final version


# TODO: Initialize LLM with temperature 0.7 for more creative writing


In [ ]:
# Exercise 3 - Step 2: Define the Three Collaborative Agents

# TODO: Define outline_agent(state: Ex3State) -> dict
#   - SystemMessage: "You are a McKinsey Engagement Manager. Create a report outline
#     using the pyramid principle: governing thought at the top, 3-4 supporting arguments,
#     evidence/analysis under each. Include section titles and key message for each section."
#   - HumanMessage: f"Create report outline for: {state['topic']}"
#   - Print: "  [EM] Outline created"
#   - Return: {"outline": response.content}


# TODO: Define content_agent(state: Ex3State) -> dict
#   - SystemMessage: "You are a McKinsey Associate. Develop full content following
#     the outline exactly. For each section, provide: analysis, data points,
#     evidence, and mini-conclusions. Use clear consulting language."
#   - HumanMessage: Include BOTH topic AND outline from state
#   - Print: f"  [Associate] Draft complete ({len(response.content)} chars)"
#   - Return: {"draft": response.content}


# TODO: Define review_agent(state: Ex3State) -> dict
#   - SystemMessage: "You are a McKinsey Senior Partner. Review and polish this report
#     for board-level readability. Sharpen the 'so what' of each section,
#     ensure recommendations are actionable, add strategic perspective.
#     The final output should be ready for a Fortune 500 board presentation."
#   - HumanMessage: state["draft"]
#   - Print: f"  [Partner] Report polished ({len(response.content)} chars)"
#   - Return: {"polished": response.content}


In [ ]:
# Exercise 3 - Step 3: Build Graph + Run

# TODO: Build StateGraph with 3 nodes in sequence:
#   outline_agent --> content_agent --> review_agent --> END


# TODO: Test with this topic:
# result = app.invoke({
#     "topic": "Market entry assessment for a US healthcare company expanding into preventive care in Southeast Asia",
#     "outline": "", "draft": "", "polished": ""
# })
# print(f"\nBoard-Ready Report:\n{result['polished'][:500]}...")


### Expected Output

You should see three collaborative steps:

```
  [EM] Outline created
  [Associate] Draft complete (4500 chars)
  [Partner] Report polished (5200 chars)

Board-Ready Report:
# Market Entry Assessment: US Healthcare Company -- Preventive Care in Southeast Asia

## Governing Thought
Southeast Asia's preventive care market represents a $12B opportunity by 2028...

## Section 1: Market Opportunity is Compelling and Timing is Right
...
```

Key verification:
- The outline should follow pyramid principle (answer first, then support)
- The draft should follow the outline's structure exactly
- The polished version should read as board-ready with sharper language and clearer 'so what'

---
## Optional Exercises

Complete these if you finish the required exercises early. They combine multiple patterns and are progressively at an intermediate level.

---
### Optional Exercise 1 (Intermediate): Intent-Based Router with 4 Specialists

**Reference:** Demo 5 in the demos notebook


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Scenario:** Build a client request routing system with:
- **Intent Classifier** -- classifies requests into 4 categories: `financial`, `strategic`, `operational`, `technical`
- **4 Specialist Agents** -- each handles their domain with appropriate expertise
- **Quality Review** -- scores the output 1-10 and provides feedback

**Architecture:**
```
[intent_classifier] --conditional--> [financial_specialist]  --> [quality_review] --> END
                    \--conditional--> [strategic_specialist]  --> [quality_review] --> END
                    \--conditional--> [operational_specialist]--> [quality_review] --> END
                    \--conditional--> [technical_specialist]  --> [quality_review] --> END
```

**Hints:**
- The intent classifier should respond with exactly one of: `financial`, `strategic`, `operational`, `technical`
- The quality review should output JSON: `{"score": N, "feedback": "..."}`
- Test with 4 diverse requests to verify all routes work

In [ ]:
# Optional Exercise 1: Intent-Based Router with 4 Specialists

# TODO: Define Opt1State TypedDict:
#   - user_input, intent, specialist_output, quality_score, final_output


# TODO: Initialize LLM with temperature 0


# TODO: Define intent_classifier -- classifies into 4 categories


# TODO: Define financial_specialist -- handles financial questions


# TODO: Define strategic_specialist -- handles strategy questions


# TODO: Define operational_specialist -- handles operations questions


# TODO: Define technical_specialist -- handles technology/data/AI questions


# TODO: Define quality_review -- scores output 1-10 with JSON feedback


# TODO: Define route_to_specialist function


# TODO: Build StateGraph with conditional edges from classifier + regular edges to quality


# TODO: Test with 4 requests:
# test_requests = [
#     "What is the NPV of investing $200M in a new manufacturing plant with 12% cost of capital?",
#     "Should we enter the Indian electric vehicle market given current competitive dynamics?",
#     "How do we reduce order-to-delivery cycle time from 14 days to 3 days?",
#     "Recommend a cloud migration strategy for our legacy SAP system"
# ]
# for req in test_requests:
#     print(f"\nRequest: {req}")
#     r = app.invoke({"user_input": req, "intent": "", "specialist_output": "", "quality_score": "", "final_output": ""})
#     print(f"  Intent: {r['intent']}")
#     print(f"  QA: {r['quality_score'][:80]}")


---
### Optional Exercise 2 (Intermediate): Parallel Analysis with Synthesis

**Reference:** Demo 3 in the demos notebook


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Scenario:** Build a due diligence system with 3 parallel analysis workstreams that feed into a synthesis step:
- **Market Analyst** -- assesses market size, growth trends, regulatory landscape
- **Financial Analyst** -- evaluates revenue quality, margins, cash flow, valuation
- **Competitive Analyst** -- maps competitive positioning, moats, threats
- **Synthesis Agent** -- combines all three into an executive recommendation

**Architecture:**
```
[market] --> [financial] --> [competitive] --> [synthesis] --> END
```

(Sequential execution, but the pattern is fan-out/fan-in because agents are independent)

**Key requirement:** The synthesis agent must:
- State a clear recommendation: PROCEED, PROCEED WITH CAUTION, or PASS
- List top 3 risks and top 3 opportunities
- Propose 2-3 critical next steps

In [ ]:
# Optional Exercise 2: Parallel Analysis with Synthesis

# TODO: Define Opt2State TypedDict:
#   - deal_description, market_analysis, financial_analysis, competitive_analysis, synthesis_report


# TODO: Initialize LLM


# TODO: Define market_analyst -- assesses TAM, growth, regulation
#   - Should output 3-4 key findings
#   - Print: "  [Market Analysis] Complete"


# TODO: Define financial_analyst -- evaluates revenue, margins, valuation
#   - Should output 3-4 key findings
#   - Print: "  [Financial Analysis] Complete"


# TODO: Define competitive_analyst -- maps positioning, moats, threats
#   - Should output 3-4 key findings
#   - Print: "  [Competitive Analysis] Complete"


# TODO: Define synthesis_agent -- combines all into executive recommendation
#   - Must include: recommendation (PROCEED/CAUTION/PASS), top 3 risks, top 3 opportunities, next steps
#   - Input should include ALL three analysis outputs


# TODO: Build graph: market --> financial --> competitive --> synthesis --> END


# TODO: Test with:
# result = app.invoke({
#     "deal_description": "Potential acquisition of a Series C fintech company providing embedded lending APIs to e-commerce platforms. $45M ARR, 40% YoY growth, -15% net margin, 200+ enterprise clients.",
#     "market_analysis": "", "financial_analysis": "", "competitive_analysis": "", "synthesis_report": ""
# })
# print(f"\nSynthesis Report:\n{result['synthesis_report'][:500]}...")


---
### Optional Exercise 3 (Intermediate): Full Engagement Delivery System

**Reference:** Combines Demo 1 (supervisor), Demo 2 (handoff), and Demo 5 (intent + quality)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
**Scenario:** Build a complete engagement delivery system that combines multiple patterns:

1. **Intake** -- receives the client request
2. **Classify** -- determines request type (strategic, operational, or analytical)
3. **Route to Specialist** -- conditional routing to the right specialist
4. **Specialist Handoff Chain** -- specialist produces initial output, then a "refinement" agent improves it
5. **Quality Gate** -- scores the output; if score >= 7, output is delivered; if < 7, it gets flagged
6. **Output** -- final client-ready deliverable

**Architecture:**
```
[classify] --conditional--> [strategic_specialist]  --> [refine] --> [quality_gate] --conditional--> END (if score >= 7)
           \--conditional--> [operational_specialist]--> [refine] --> [quality_gate] \--conditional--> [flag_for_review] --> END
           \--conditional--> [analytical_specialist] --> [refine] --> [quality_gate]
```

**Key challenges:**
- You need TWO conditional edges: one after classify (routing) and one after quality (pass/flag)
- The refine agent must read the specialist's output and improve it
- The quality gate must parse a JSON score and make a routing decision

**Hints:**
- For the quality gate routing, extract the score from the JSON output and compare to 7
- The `flag_for_review` node can simply prepend a warning message to the output
- You will need at least 7 nodes in your graph

In [ ]:
# Optional Exercise 3: Full Engagement Delivery System

# TODO: Define Opt3State TypedDict:
#   - client_request (str)
#   - intent (str)
#   - specialist_output (str)
#   - refined_output (str)
#   - quality_score (str)  -- JSON string from quality agent
#   - final_output (str)
#   - flagged (str)  -- "true" or "false"


# TODO: Initialize LLM


# TODO: Define intent_classifier -- classifies as 'strategic', 'operational', or 'analytical'


# TODO: Define strategic_specialist


# TODO: Define operational_specialist


# TODO: Define analytical_specialist


# TODO: Define refinement_agent -- takes specialist_output and improves clarity/depth
#   - SystemMessage: "You are a senior McKinsey editor. Take this specialist output and
#     improve its clarity, structure, and depth. Ensure it follows the pyramid principle
#     and every statement has a clear 'so what'."
#   - Return: {"refined_output": response.content}


# TODO: Define quality_gate -- scores 1-10, outputs JSON {"score": N, "feedback": "..."}
#   - Return: {"quality_score": response.content, "final_output": state["refined_output"]}


# TODO: Define flag_for_review -- prepends warning to output
#   - Return: {"final_output": f"[FLAGGED FOR HUMAN REVIEW]\n\n{state['refined_output']}", "flagged": "true"}


# TODO: Define route_to_specialist(state) -> str
#   - Route based on state["intent"]


# TODO: Define route_after_quality(state) -> str
#   - Parse JSON from state["quality_score"], extract score
#   - If score >= 7: return "end"
#   - Else: return "flag"
#   - Wrap in try/except -- default to "flag" if parsing fails


# TODO: Build the full graph with both conditional edges
# graph = StateGraph(Opt3State)
# ... add all nodes ...
# graph.set_entry_point("classify")
# graph.add_conditional_edges("classify", route_to_specialist, {...})
# graph.add_edge("strategic_specialist", "refine")
# graph.add_edge("operational_specialist", "refine")
# graph.add_edge("analytical_specialist", "refine")
# graph.add_edge("refine", "quality_gate")
# graph.add_conditional_edges("quality_gate", route_after_quality, {
#     "end": END,
#     "flag": "flag_for_review"
# })
# graph.add_edge("flag_for_review", END)
# app = graph.compile()


# TODO: Test with 3 diverse requests:
# test_cases = [
#     "What should be our 5-year growth strategy in renewable energy?",
#     "Design a new organizational structure for our merged supply chain team",
#     "Analyze the correlation between our marketing spend and customer acquisition cost over the last 8 quarters"
# ]
# for req in test_cases:
#     print(f"\nClient: {req}")
#     r = app.invoke({
#         "client_request": req, "intent": "", "specialist_output": "",
#         "refined_output": "", "quality_score": "", "final_output": "", "flagged": "false"
#     })
#     print(f"  Intent: {r['intent']}")
#     print(f"  Flagged: {r['flagged']}")
#     print(f"  QA: {r['quality_score'][:80]}")
#     print(f"  Output: {r['final_output'][:200]}...")


### Expected Output for Optional Exercise 3

```
Client: What should be our 5-year growth strategy in renewable energy?
  Intent: strategic
  Flagged: false
  QA: {"score": 8, "feedback": "Clear strategic framework with actionable..."}
  Output: ## 5-Year Renewable Energy Growth Strategy...

Client: Design a new organizational structure for our merged supply chain team
  Intent: operational
  Flagged: false
  QA: {"score": 7, "feedback": "Solid operational recommendation with..."}
  Output: ## Post-Merger Supply Chain Organization Design...

Client: Analyze the correlation between our marketing spend and customer acquisition cost...
  Intent: analytical
  Flagged: false
  QA: {"score": 8, "feedback": "Strong quantitative framework..."}
  Output: ## Marketing Spend vs. CAC Analysis...
```

Key verification:
- All 3 intents should route correctly
- The refinement step should improve each specialist's output
- Quality scores should generally be 7+ (if < 7, the flagging mechanism should trigger)
- The system should handle all request types end-to-end without errors

---
## Summary

In these exercises, you practiced:

| Exercise | Pattern | Key Skill |
|----------|---------|----------|
| 1 (Required) | Supervisor-Worker | Conditional routing based on LLM classification |
| 2 (Required) | Agent Handoff | Context preservation across agent boundaries |
| 3 (Required) | Collaborative Build | Sequential contribution to a shared deliverable |
| Opt 1 (Intermediate) | Intent Router + QA | 4-way routing with quality gate |
| Opt 2 (Intermediate) | Parallel + Synthesis | Independent agents feeding into a merger step |
| Opt 3 (Intermediate) | Full System | Combining supervisor + handoff + quality gate |

**Key takeaways:**
- Multi-agent systems decompose complex problems into specialized, composable steps
- Context passing (via state fields) is the glue that holds agents together
- Quality gates prevent subpar outputs from reaching end users
- LangGraph's `StateGraph` + `conditional_edges` can express any of these patterns